# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [7]:
import re # regular expressions for pattern matching
import Bio.SeqIO # reading sequence files such as FASTQ
import Bio.Seq # seq objects such as reverse_complement()

# SeqIO.parse returns reads one at a time; list() pulls them all into memory
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

# 3' ends and the fixed sequence before the barcode (forward-orientation)
HA_END = "CCGGATTTGCATATAATGATGCACCAT"
NA_END = "CACGATAGATAAATAATAGTGCACCAT"
UPSTREAM = "AGGCGGCCGC"

# Define the pattern (either gene end followed by UPSTREAM) and function to find it 
gene_pattern = re.compile(f"(?P<gene_end>{HA_END}|{NA_END}){UPSTREAM}")

def classify_read(seqread):
    """Classify a read as HA or NA.

    Args:
        seqread: Nucleotide sequence read matching in reverse orientation 

    Returns:
        "HA" or "NA" 
        None if the read matches neither
    """
    if isinstance(seqread, str):
        seqread = Bio.Seq.Seq(seqread)
    forward = str(seqread.reverse_complement()) 

    match = gene_pattern.search(forward)
    if match is None:
        return None
    return "HA" if match.group("gene_end") == HA_END else "NA"

# Helper function from lecture 9
def run_tests(parser_function, test_cases):
    n_failed = 0
    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")
    return f"{n_failed} tests failed"

# Reverse complement of gene end-UPSTREAM-barcode
def rc(s):
    return str(Bio.Seq.Seq(s).reverse_complement())

barcode = "ACGTACGTACGTACGT"
test_cases = [
    (rc(HA_END + UPSTREAM + barcode), "HA", "positive: HA read"),
    (rc(NA_END + UPSTREAM + barcode), "NA", "positive: NA read"),
    (rc(HA_END + UPSTREAM + "ACGTACGTACGTACGY"), "HA", "HA read with invalid barcode is still HA"),
    (rc("T" * 27 + UPSTREAM + barcode), None, "negative: neither gene end"),
    (rc(HA_END + "AGGCGGCGGC" + barcode), None, "negative: wrong upstream sequence"),
]
print(run_tests(classify_read, test_cases))

# Classify every read and count
gene_counts = {"HA": 0, "NA": 0, "neither": 0}

for read in seqreads:
    gene = classify_read(read.seq)
    if gene is None:
        gene_counts["neither"] += 1
    else:
        gene_counts[gene] += 1

print(f"HA reads: {gene_counts['HA']}")
print(f"NA reads: {gene_counts['NA']}")
print(f"Neither:  {gene_counts['neither']}")

PASS  positive: HA read
PASS  positive: NA read
PASS  HA read with invalid barcode is still HA
PASS  negative: neither gene end
PASS  negative: wrong upstream sequence
0 tests failed
HA reads: 5316
NA reads: 3973
Neither:  711


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [8]:
def read_barcode(seqread, bclen=16, upstream=UPSTREAM):
    """Identify a barcode with a known upstream sequence.

    Args:
        sseqread: Nucleotide sequence read matching in reverse orientation 
        bclen: Barcode length
        upstream: Nucleotide sequence upstream barcode in forward orientation. 

    Returns:
        Barcode in the forward orientation
        None if the read does not have a valid barcode
    """
    forward = str(Bio.Seq.Seq(str(seqread)).reverse_complement()).upper()

    # Define the pattern (UPSTREAM followed by barcode, with specifications on the length of the barcode) 
    pattern = re.compile(upstream + f"(?P<barcode>[ACGTN]{{{bclen}}})$")

    match = pattern.search(forward)
    if match is None:
        return None
    return match.group("barcode")


barcode_counts = {"HA": {}, "NA": {}}
invalid_barcodes = {"HA": 0, "NA": 0}

for read in seqreads:
    gene = classify_read(read.seq)
    if gene is None:
        continue 

    barcode = read_barcode(read.seq)
    if barcode is None:
        invalid_barcodes[gene] += 1          # HA or NA read with no valid barcode
    elif barcode in barcode_counts[gene]:
        barcode_counts[gene][barcode] += 1   # barcode seen before
    else:
        barcode_counts[gene][barcode] = 1    # first time we see this barcode

print(f"HA reads without a valid barcode: {invalid_barcodes['HA']}")
print(f"NA reads without a valid barcode: {invalid_barcodes['NA']}")

HA reads without a valid barcode: 70
NA reads without a valid barcode: 63


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [9]:
# Use barcode_counts["HA"] and barcode_counts["NA"] each map barcode from question 2 

for gene in ["HA", "NA"]:
    top_barcode = None  
    top_count = 0       

    for barcode, count in barcode_counts[gene].items():
        if count > top_count:       
            top_barcode = barcode
            top_count = count

    print(f"{gene}: The barcode with the most counts is {top_barcode} with {top_count} counts")

HA: The barcode with the most counts is CCCGACCCGACATTAA with 155 counts
NA: The barcode with the most counts is ACCAGTTCTCCCCGGG with 152 counts
